In [ ]:
# ==========================================================================
# 자동차 부품 제조 공정 설명 키워드 분석 웹 화면 (Gradio) — 셀 하나로 실행
# --------------------------------------------------------------------------
# [전체 흐름]
#   0. 도구 설치            : kiwipiepy, koreanize-matplotlib, wordcloud, gradio
#   1. 전처리 함수          : 반복 문구(사용장비·품질기준·특이사항) 지우기 → 잘림(...)·깨진 기호·숫자 정리
#                             → Kiwi로 명사·외국어 뽑기 → 불용어 빼기·표기 통일
#   2. 분석 함수 (analyze)  : 파일 읽기 → 전처리 → 빈도 3종(등장 횟수·문서 빈도·TF-IDF) 비교 표
#                             → 워드클라우드 → 2-gram TOP 15 → 공정 단계별 언급 비율
#   3. 질문 함수 (ask)      : 질문(키워드나 문장) → 같은 방법으로 키워드 뽑기 → 키워드별 빈도·순위·이웃 키워드(대답)
#                             → TF-IDF 유사도로 관련 공정 설명 5개 찾기(검색 결과)
#   4. 웹 화면 실행         : ① 파일 선택 → ② 분석 결과 → ③ 질문하기 순서로 배치
#
# [사용 순서]
#   ① 이 셀 실행 → 셀 아래(또는 https://….gradio.live 주소)에 웹 화면이 나타남
#   ② 공정 설명 .csv 파일을 고르고 → [분석하기] (20초 정도)
#   ③ 키워드(예: 용접)나 문장(예: 도장 전에 세척하는 공정)을 입력하고 → [질문하기] (Enter 가능)
#   끝낼 때: 셀 왼쪽의 ■(중지) 버튼
#
# [주의] 공개 주소(….gradio.live)는 주소를 아는 누구나 접속할 수 있음
# ==========================================================================


# ----- 0. 도구 설치 -----
# PIL(Pillow): Colab에 이미 설치된 이미지 도구, 지금 버전 번호를 알아내려고 불러옴
import PIL

# pip 설치 (-q: 메시지 줄이기)
# pillow=={PIL.__version__}: 설치 중에 Pillow 버전이 바뀌어 ImportError가 나는 것을 막으려고 지금 버전으로 고정
!pip install -q kiwipiepy koreanize-matplotlib wordcloud gradio pillow=={PIL.__version__}


# ----- 도구 불러오기 -----
import os                                         # 파일 경로 다루기 (워드클라우드 글꼴 위치)
import re                                         # 정규식(글자 패턴 찾기·바꾸기)
from collections import Counter                   # 개수 세기
import pandas as pd                               # 표(DataFrame) 다루기
import matplotlib.pyplot as plt                   # 그래프 그리기
import koreanize_matplotlib                       # 불러오기만 하면 그래프 한글 글꼴(나눔고딕)이 설정됨
from wordcloud import WordCloud                   # 워드클라우드
from kiwipiepy import Kiwi                        # 한국어 형태소 분석기
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer   # TF-IDF, 단어 묶음 세기
from sklearn.metrics.pairwise import cosine_similarity                         # 문서끼리 비슷한 정도(0~1)
import gradio as gr                               # 웹 화면 만들기

# 한국어 형태소 분석기 준비 (처음 한 번 몇 초 걸림)
kiwi = Kiwi()

# koreanize_matplotlib 안에 들어 있는 나눔고딕 글꼴 파일 위치 (워드클라우드는 한글 글꼴 경로를 따로 알려 줘야 함)
FONT_PATH = os.path.join(os.path.dirname(koreanize_matplotlib.__file__), "fonts", "NanumGothic.ttf")


# ----- 1. 전처리 함수 -----
# ✏️ 불용어: 거의 모든 공정 설명에 나와서 구별에 도움이 안 되는 말
STOP = {"제품", "공정", "부품", "생산", "자동차", "작업", "이용", "사용", "완료", "처리", "실시", "진행",
        "필요", "이상", "부분", "상태", "종류", "형태", "경우", "해당", "위해", "op"}

# ✏️ 표기 통일: {바꿀 말: 통일할 말}
NORM = {"원재료": "원자재", "원료": "원자재", "출고": "출하", "구리스": "그리스"}

# ✏️ 공정 단계 사전: {단계: [그 단계를 뜻하는 키워드들]}
STAGES = {
    "입고·투입": ["입고", "원자재", "자재", "투입", "계량", "발주", "조달"],
    "가공·성형": ["가공", "절단", "성형", "사출", "프레스", "절삭", "단조", "주조", "밴딩", "타발", "재단", "cnc"],
    "접합·조립": ["조립", "용접", "압입", "체결", "융착", "부착", "삽입", "장착"],
    "표면처리": ["도장", "세척", "건조", "도금", "탈지", "코팅", "열처리", "도포", "사상"],
    "검사": ["검사", "측정", "시험", "테스트", "외관", "검수", "육안", "불량"],
    "포장·출하": ["포장", "출하", "납품", "적재", "박스", "보관", "창고"],
}


# 글 하나 → 정리된 문장
def clean(text):
    text = re.sub(r"/\s*(사용장비|품질기준|특이사항):[^/]*", " ", text)   # 반복 문구 지우기
    text = re.sub(r"작업 수행", " ", text)                              # 반복 문구 앞에 붙던 말 지우기
    text = re.sub(r"\.\.\.|\bgt\b", " ", text)                         # 잘림 표시(...), 깨진 기호(&gt;의 gt) 지우기 (\b: 단어 경계)
    text = re.sub(r"[^가-힣A-Za-z]", " ", text)                         # 한글·영어가 아닌 글자(숫자, 기호, 괄호) → 공백
    return re.sub(r"\s+", " ", text).strip()                           # 여러 칸 공백을 한 칸으로, 앞뒤 공백 지우기


# 문장 하나 → 키워드 목록
#   t.tag in (...): 일반명사·고유명사·외국어만,  t.form.lower(): 영어는 소문자로 (CNC와 cnc를 같은 말로)
#   NORM.get(w, w): 사전에 있으면 통일할 말로,   len(w) >= 2: 한 글자짜리는 뜻이 불분명해서 뺌
def to_keywords(sentence):
    words = [t.form.lower() for t in kiwi.tokenize(sentence) if t.tag in ("NNG", "NNP", "SL")]
    words = [NORM.get(w, w) for w in words]
    return [w for w in words if len(w) >= 2 and w not in STOP]


# CSV 파일 읽기: utf-8로 먼저 읽어 보고, 한글이 깨져 오류가 나면 cp949(공공기관·엑셀 파일에 흔한 방식)로 다시 읽기
def read_csv(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


# ----- 2. 분석 함수: [분석하기] 버튼 -----
def analyze(csv_file):
    # 파일을 고르지 않았으면 안내 메시지만 돌려줌 (나머지 출력은 비움)
    if csv_file is None:
        return "공정 설명 .csv 파일을 선택하세요.", None, None, None, None, None

    # 파일 읽기 → 공정명과 공정설명을 이어 붙여 정리 → 키워드 목록 (10초 정도 걸림)
    df = read_csv(csv_file)
    df["문장"] = (df["공정명"].fillna("") + " " + df["공정설명"].fillna("")).apply(clean)
    df["키워드"] = df["문장"].apply(to_keywords)
    df["키워드문자열"] = df["키워드"].apply(" ".join)         # 띄어쓰기로 이은 글자 (TF-IDF 입력용)

    # 빈도 3종
    #   ① 등장 횟수: 모든 행의 키워드를 펼쳐서 세기
    #   ② 문서 빈도: 행마다 set()으로 중복을 없앤 뒤 세기 → '몇 개 행에 나왔나'
    #   ③ TF-IDF 합계: 행마다 TF-IDF 점수를 구한 뒤 키워드별로 모든 행에 걸쳐 더하기 (.A1: 1줄짜리 배열로)
    count_all = Counter(w for kws in df["키워드"] for w in kws)
    count_doc = Counter(w for kws in df["키워드"] for w in set(kws))
    tfidf = TfidfVectorizer(analyzer=str.split)               # analyzer=str.split: 이미 나눠 둔 키워드 그대로
    X = tfidf.fit_transform(df["키워드문자열"])               # (행 수, 키워드 수) 점수 표
    tfidf_sum = pd.Series(X.sum(axis=0).A1, index=tfidf.get_feature_names_out())

    # 세 방식의 상위 15개를 나란히 놓은 표 (most_common: 많은 순서로, nlargest: 큰 순서로)
    compare = pd.DataFrame({
        "순위": range(1, 16),
        "등장 횟수": [f"{w} ({n})" for w, n in count_all.most_common(15)],
        "문서 빈도(행 수)": [f"{w} ({n})" for w, n in count_doc.most_common(15)],
        "TF-IDF 합계": [f"{w} ({s:.1f})" for w, s in tfidf_sum.nlargest(15).items()],
    })

    # 워드클라우드 (문서 빈도 기준, max_words: 최대 키워드 수, random_state: 매번 같은 배치)
    wc = WordCloud(font_path=FONT_PATH, width=900, height=450, background_color="white",
                   max_words=80, colormap="viridis", random_state=42).generate_from_frequencies(count_doc)
    fig_wc, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(wc)
    ax.axis("off")                                         # 눈금·테두리 숨기기
    plt.close(fig_wc)                                      # 셀 아래에 따로 그려지지 않게 닫기 (웹 화면에만 표시)

    # 2-gram: 키워드 목록에서 바로 옆에 붙은 두 키워드 묶음 세기
    #   token_pattern=r"\S+": 띄어쓰기로 나뉜 덩어리를 한 단어로, ngram_range=(2, 2): 두 단어 묶음만
    bigram = CountVectorizer(token_pattern=r"\S+", ngram_range=(2, 2))
    B = bigram.fit_transform(df["키워드문자열"])
    bigram_count = pd.Series(B.sum(axis=0).A1, index=bigram.get_feature_names_out())
    top_bigram = bigram_count.nlargest(15)
    fig_bi, ax = plt.subplots(figsize=(7, 5))
    ax.barh(top_bigram.index[::-1], top_bigram.values[::-1], color="#2a78d6")   # [::-1]: 큰 값이 위로
    ax.bar_label(ax.containers[0], fontsize=8)             # 막대 끝에 숫자 표시
    ax.set_title("붙어 나오는 두 키워드 TOP 15")
    plt.tight_layout()
    plt.close(fig_bi)

    # 공정 단계별로 '그 단계 키워드가 하나라도 나온 행'의 비율(%)
    #   any(w in words for w in kws): 행의 키워드 중 하나라도 단계 사전에 있으면 True → mean(): True 비율
    stage_ratio = pd.Series({stage: df["키워드"].apply(lambda kws: any(w in words for w in kws)).mean() * 100
                             for stage, words in STAGES.items()})
    fig_st, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(stage_ratio.index[::-1], stage_ratio.values[::-1], color="#1baf7a")
    ax.bar_label(ax.containers[0], fmt="%.0f%%", fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_xlabel("언급된 행 비율 (%)")
    ax.set_title(f"공정 단계별 언급 비율 ({len(df)}행 중)")
    plt.tight_layout()
    plt.close(fig_st)

    # 처리 결과 메시지
    message = (f"공정 {len(df):,}개 분석 완료 / 서로 다른 키워드 {len(count_all):,}개 / "
               f"행당 키워드 평균 {df['키워드'].apply(len).mean():.1f}개")

    # 보관함: 질문 함수가 쓸 도구와 데이터 묶음
    store = (df, count_all, count_doc, tfidf, X, bigram_count)
    return message, compare, fig_wc, fig_bi, fig_st, store


# ----- 3. 질문 함수: [질문하기] 버튼 -----
def ask(question, store):
    # 분석 전이거나 질문이 비어 있으면 안내 메시지
    if store is None:
        return "먼저 파일을 고르고 [분석하기]를 누르세요.", None
    if not question.strip():
        return "키워드나 문장을 입력하세요.", None

    # 보관함에서 데이터와 도구 꺼내기
    df, count_all, count_doc, tfidf, X, bigram_count = store

    # 질문도 같은 방법으로 정리 → 키워드 뽑기 (dict.fromkeys: 순서를 지키면서 중복 없애기)
    q_words = list(dict.fromkeys(to_keywords(clean(question))))
    if not q_words:
        return "질문에서 분석할 키워드를 찾지 못했습니다. 명사로 된 키워드(예: 용접, 도장)를 넣어 보세요.", None

    # 문서 빈도 순위표 (1위부터): {키워드: 순위}
    rank = {w: i + 1 for i, (w, _) in enumerate(count_doc.most_common())}

    # 키워드마다 대답 한 줄씩 만들기
    lines = []
    for w in q_words:
        if w not in count_doc:                                       # 데이터에 한 번도 안 나온 키워드
            lines.append(f"• {w}: 공정 설명에 나오지 않는 키워드입니다.")
            continue
        stage = [s for s, words in STAGES.items() if w in words]      # 공정 단계 사전에서 속한 단계 찾기
        # 이웃 키워드: 2-gram 중 이 키워드가 앞이나 뒤에 있는 묶음 → 상대 키워드별로 횟수 합치기 → 상위 5개
        pairs = bigram_count[[w in g.split() for g in bigram_count.index]]
        neighbors = Counter()
        for g, n in pairs.items():
            a, b = g.split()
            other = b if a == w else a
            if other != w:                                           # '검사 검사' 같은 같은 말 반복은 빼기
                neighbors[other] += n
        lines.append(
            f"• {w}: {count_doc[w]}개 행({count_doc[w] / len(df):.0%})에 나옴, 문서 빈도 {rank[w]}위, "
            f"전체 {count_all[w]}번 등장"
            + (f", 공정 단계 '{stage[0]}'" if stage else "")
            + (f"\n   함께 붙어 나온 키워드: {', '.join(f'{o}({n})' for o, n in neighbors.most_common(5))}" if neighbors else ""))
    answer = f"질문 키워드: {', '.join(q_words)}\n" + "\n".join(lines)

    # 검색 결과: 질문 키워드와 TF-IDF 코사인 유사도가 가장 높은 공정 5개
    scores = cosine_similarity(tfidf.transform([" ".join(q_words)]), X)[0]   # 행마다 비슷한 정도(0~1)
    top = scores.argsort()[::-1][:5]                                   # 유사도 큰 순서로 5개 위치
    top = top[scores[top] > 0]                                         # 겹치는 키워드가 없는(유사도 0) 행은 빼기
    result = df.loc[top, ["공정도명", "공정순서", "공정명", "문장"]].assign(유사도=scores[top].round(3))
    result["공정도명"] = result["공정도명"].str.replace(" 제조업 생산공정", "")   # 표가 좁아지게 공통 꼬리말 빼기
    result["문장"] = result["문장"].str[:150]                           # 긴 문장은 앞 150글자만
    return answer, result


# ----- 4. 웹 화면 -----
# gr.Blocks(): 웹 화면 전체 틀, with 블록 안의 부품이 위에서부터 차례로 배치됨
with gr.Blocks() as demo:
    gr.Markdown("# 🏭 자동차 부품 제조 공정 설명 키워드 분석")
    gr.Markdown("공정 설명 글에서 키워드를 뽑아 **빈도 3종 비교 · 워드클라우드 · 2-gram · 공정 단계별 언급 비율**을 보여 주고, "
                "키워드를 물어보면 빈도·순위·함께 나오는 키워드를 대답하고 관련 공정 설명을 찾아 줍니다. (Kiwi + TF-IDF)")

    # 보이지 않는 보관함: analyze가 만든 (데이터, 빈도, TF-IDF, 2-gram)을 담아 ask에 넘겨줌
    store = gr.State()

    # ===== ① 파일 선택 =====
    gr.Markdown("## ① 파일 선택")
    csv_file = gr.File(label="공정 설명 파일 (.csv, 열: 공정명, 공정설명 …)", file_types=[".csv"])
    analyze_btn = gr.Button("분석하기", variant="primary")
    status = gr.Textbox(label="처리 결과")

    # ===== ② 분석 결과 =====
    gr.Markdown("## ② 분석 결과")
    compare = gr.Dataframe(label="빈도 3종 비교 TOP 15 (등장 횟수 · 문서 빈도 · TF-IDF 합계)")
    wc_plot = gr.Plot(label="워드클라우드 (문서 빈도)")
    with gr.Row():                                         # gr.Row(): 안의 부품을 가로로 나란히
        bigram_plot = gr.Plot(label="2-gram")
        stage_plot = gr.Plot(label="공정 단계별 언급 비율")

    # ===== ③ 질문하기 =====
    gr.Markdown("## ③ 질문하기")
    question = gr.Textbox(label="질문 (키워드나 문장)", placeholder="예) 용접  /  도장 전에 세척하는 공정")
    # ✏️ 예시 질문: 누르면 입력 칸이 채워짐
    gr.Examples(examples=["용접", "도장 전에 세척하는 공정", "외관 검사 후 포장", "사출 성형", "CNC 가공"], inputs=question)
    ask_btn = gr.Button("질문하기", variant="primary")
    answer = gr.Textbox(label="대답", lines=6)
    result = gr.Dataframe(label="검색 결과: 관련 공정 설명 5개", wrap=True,
                          column_widths=["16%", "8%", "18%", "50%", "8%"])   # 열 너비 (공정도명, 공정순서, 공정명, 문장, 유사도)

    # 버튼과 함수 연결: 함수가 돌려준 값이 outputs 부품에 순서대로 표시됨
    analyze_btn.click(analyze, inputs=csv_file, outputs=[status, compare, wc_plot, bigram_plot, stage_plot, store])
    ask_btn.click(ask, inputs=[question, store], outputs=[answer, result])
    question.submit(ask, inputs=[question, store], outputs=[answer, result])   # Enter로도 실행

# 웹 화면 실행 (debug=True: 셀이 계속 실행되며 오류를 셀 아래에 보여 줌)
demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 7.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 60.4 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6425a5453ac650afca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (http